This notebook fine-tunes `google-bert/bert-base-uncased` on the following tasks:

| Task | Description | Metric(s) |
|------|-------------|----------|
| **SST-2** | Sentiment classification | Accuracy |
| **MRPC** | Paraphrase detection | Accuracy and F1 |
| **SQuAD** | Question Answering | Exact Match (EM) and F1 |


**Reference performance:**
- SST-2: **93.5% Accuracy**
- MRPC: **88.9% F1**
- Squad：**80.8% EM**  &  **88.5% F1**

---

---
## Setup

We installed all required Python packages and import them. We loaded the pre-trained BERT tokenizer and model. And visualize the WordPiece tokenization process on an example sentence.

In [ ]:

!pip install -q transformers datasets evaluate torch pandas matplotlib seaborn scikit-learn

In [ ]:
# ── Part 1: Imports ──────────────────────────────────────────────────────────
import torch
import collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset
import evaluate

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

c:\Users\Super\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version : 2.10.0+cpu
CUDA available  : False


In [ ]:
# ── Part 1: Load AutoTokenizer ───────────────────────────────────────────────

MODEL_CHECKPOINT = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f"Tokenizer loaded: {MODEL_CHECKPOINT}")

# Visualize word_piece
example_sentence = "Tokenization is incredibly fascinating!"
print(f"\nOriginal sentence: {example_sentence}\n")

print(f"{'-'*40}")
print(f"{'Word':<15} | {'WordPiece Tokens'}")
print(f"{'-'*40}")

# Split by space to get individual words, then tokenize each to see the breakdown
for word in example_sentence.split():

    # expected output formate
    """
    ----------------------------------------
    Word            | WordPiece Tokens
    ----------------------------------------
    Tokenization    | ['token', '##ization']
    is              | ['is']
    incredibly      | ['incredibly']
    fascinating!    | ['fascinating', '!']
    ----------------------------------------
    """

    # implenment your code here
    a = tokenizer.tokenize(word)
    print(f"{word:<15} | {a}")

print(f"{'-'*40}")




Tokenizer loaded: google-bert/bert-base-uncased

Original sentence: Tokenization is incredibly fascinating!

----------------------------------------
Word            | WordPiece Tokens
----------------------------------------
Tokenization    | ['token', '##ization']
is              | ['is']
incredibly      | ['incredibly']
fascinating!    | ['fascinating', '!']
----------------------------------------


---
## Part 2 — Data Loading & Inspection

Load sst2 and mrpc datasets and print one example from each to confirm column names and data format.

In [ ]:
# ── Part 2: Load GLUE datasets ───────────────────────────────────────────────

sst2 = load_dataset("glue", "sst2")
# implenment your code for mrpc/squad
mrpc = load_dataset("glue", "mrpc")
squad = load_dataset("squad")

print("SST-2 dataset:", sst2)
# implement your code here
print("MRPC dataset:", mrpc)
print("SQuAD dataset:", squad)

print("MRPC dataset:", mrpc)
print(mrpc['train'][0])

print("SQuAD dataset:", squad)
print(squad['train'][0])

print("SST-2 dataset:", sst2)
print(sst2['train'][0])

SST-2 dataset: DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})
MRPC dataset: DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})
SQuAD dataset: DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})
MRPC dataset: D

In [ ]:
# ── Part 2: Inspects one example from each dataset ────────────────────────────

print("=" * 60)
print("SST-2 example (input column: 'sentence')")
print("=" * 60)
# implenment your code here
print(sst2['train'][0])


print()
print("=" * 60)
print("MRPC example (input columns: 'sentence1', 'sentence2')")
print("=" * 60)
# implenment your code here
print(mrpc['train'][0])

print()
print("=" * 60)
print("SQuAD example (input columns: 'question', 'context', 'answers')")
print("=" * 60)
# implenment your code here
print(squad['train'][0])


SST-2 example (input column: 'sentence')
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}

MRPC example (input columns: 'sentence1', 'sentence2')
{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .', 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .', 'label': 1, 'idx': 0}

SQuAD example (input columns: 'question', 'context', 'answers')
{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of 

**Expected output formate**

============================================================
SST-2 example (input column: 'sentence')
============================================================
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}

============================================================
MRPC example (input columns: 'sentence1', 'sentence2')
============================================================
{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .', 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .', 'label': 1, 'idx': 0}

============================================================
SQuAD example (input columns: 'question', 'context', 'answers')
============================================================
{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.', 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?', 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}

---
## Tokenization

Implement tokenization for all three tasks. SST-2 should tokenize a single sentence, MRPC should tokenize a sentence pair, and SQuAD requires specific Question Answering mapping (handling strides and offsets).

✅ We implemented the tokenization for the MRPC task.

In [ ]:
# ── Part 3: Tokenization ─────────────────────────────────────────────────────
#   "SST-2 should tokenize a single sentence"
#   "MRPC should tokenize a sentence pair"
#   "SQuAD requires specific Question Answering mapping"

def tokenize_sst2(examples):
    """Tokenize a single sentence for SST-2 sentiment classification."""
    return tokenizer(
        examples["sentence"],       # single sentence — SST-2 column name
        truncation=True,
        padding="max_length",
        max_length=128,
    )


def tokenize_mrpc(examples):
    """Tokenize a sentence pair for MRPC paraphrase detection."""
    return tokenizer(
       examples["sentence1"],
       examples["sentence2"],
       truncation=True,
       padding="max_length",
       max_length=128,
    )

# Apply tokenization to all splits
sst2_encoded = sst2.map(tokenize_sst2, batched=True)
mrpc_encoded  = mrpc.map(tokenize_mrpc,  batched=True)

# Set format for PyTorch
sst2_encoded.set_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids", "label"])
mrpc_encoded.set_format(type="torch",  columns=["input_ids", "attention_mask", "token_type_ids", "label"])

print("SST-2 tokenized columns:", sst2_encoded["train"].column_names)
print("MRPC tokenized columns :", mrpc_encoded["train"].column_names)





# QA Preprocessing for SQuAD
max_length = 384
stride = 128

# We use this to create questions and we put it all on a list
# We shorten and then create the pieces, then we use these pieces
def preprocess_squad(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        # We link these tokens back to its first char position.
        # We finish this by setting padding to max_length which
        # puts the empty tokens and have it fill it up so that we have
        # the set amount of tokens.
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
# We start by deleting the maps used on the original data
# I put these and set these into variables.
# We use variables of start positions and end positions and store them
# into empty lists to store the end numbers of the output tokens.
    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []
# We used a for loop to calculate the answer by reading the characters
# We then use this to use the inputs.
# We have it give us a bunch of zeroes and ones then we see through the zeroes
# to see where a certain i index slot is when it first starts.

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1
    # if offeset doesn't read the character, then the start and end positions would
    # both /append a zero.
    # or else....we go read one token then the next token to see where it starts.
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)
        # We did the same thing:
        # We go read one token then the next token to see where it finishes this time.
        # We filter it back to the original dataset
            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

print("\nTokenizing SQuAD dataset...")
squad_train_encoded = squad["train"].map(
    preprocess_squad,
    batched=True,
    remove_columns=squad["train"].column_names
)

SST-2 tokenized columns: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask']
MRPC tokenized columns : ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask']

Tokenizing SQuAD dataset...


---
## Metric Functions
We implemented two metric functions: one for SST-2 accuracy and one for MRPC accuracy plus F1.

Each function receives `EvalPrediction` from the Trainer and returns a dictionary of metrics.

In [ ]:
# ── Part 4: Metric functions ─────────────────────────────────────────────────
#   "one for SST-2 accuracy"
#   "one for MRPC accuracy + F1"
#   "SQuAD requires post-processing for Exact Match and F1"

import collections
from tqdm.auto import tqdm
import numpy as np

# Load evaluate metrics
import evaluate
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
squad_metric = evaluate.load("squad")

def compute_metrics_sst2(eval_pred):
    """Compute accuracy for SST-2 using custom names."""
    scores, values = eval_pred
    guess = np.argmax(scores, axis=-1)

    return accuracy_metric.compute(predictions=guess, references=values)

def compute_metrics_mrpc(eval_pred):
    """Compute accuracy AND F1 for MRPC using custom names."""
    scores, values = eval_pred
    guess = np.argmax(scores, axis=-1)

    success = accuracy_metric.compute(predictions=guess, references=values)["accuracy"]
    f1 = f1_metric.compute(predictions=guess, references=values)["f1"]

    return {"accuracy": success, "f1": f1}



def postprocess_qa_predictions(examples, features, raw_predictions, n_best_size=20, max_answer_length=30):
    """Post-process SQuAD predictions to text answers for metric computation."""
    all_start_logits, all_end_logits = raw_predictions
    # Map features back to original examples
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, feature in enumerate(features):
        features_per_example[example_id_to_index[feature["example_id"]]].append(i)

    predictions = []

    for example_index, example in enumerate(tqdm(examples, desc="Post-processing predictions")):
        feature_indices = features_per_example[example_index]
        valid_answers = []
        context = example["context"]

        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]

            # Top N highest logits
            start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
            end_indexes = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()

            for i in start_indexes:
                for j in end_indexes:
                    if i >= len(offset_mapping) or j >= len(offset_mapping):
                        continue
                    if offset_mapping[i] is None or offset_mapping[j] is None:
                        continue
                    if j < i or j - i + 1 > max_answer_length:
                        continue
                    p1 = offset_mapping[i][0]
                    p2 = offset_mapping[j][1]

                    valid_answers.append(
                        {
                            "score": start_logits[i] + end_logits[j],
                            "text": context[p1: p2]
                        })

        if len(valid_answers) > 0:
            best_answer = sorted(valid_answers, key=lambda x: x["score"], reverse=True)[0]
        else:
            # Fallback if no valid answers found
            best_answer = {"text": "", "score": 0.0}

        predictions.append({"id": example["id"], "prediction_text": best_answer["text"]})

    return predictions

print("Metric functions defined:")
print("  compute_metrics_sst2 → accuracy")
print("  compute_metrics_mrpc → accuracy + f1")

Metric functions defined:
  compute_metrics_sst2 → accuracy
  compute_metrics_mrpc → accuracy + f1


---
## Fine-tune BERT on SST-2
We fine-tune BERT on SST-2 using the baseline hyperparameters:  
`num_train_epochs=3, learning_rate=2e-5, per_device_train_batch_size=32, and seed=42`.  

In [ ]:
# ── Part 5: Fine-tune on SST-2 ───────────────────────────────────────────────

model_sst2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,)
# We created and stored a folder where we will have certain data that we get back
# prior to when we finish training the epochs
training_args_sst2 = TrainingArguments(
    output_dir="./results_sst2",
    num_train_epochs=3,
    # We use 2e-5 to make it a certain size after calculating the weights
    # We set batch size in the sentences so that it knows how much it reads
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    # We set seed to 42 to make sure we get consistent results
    # We see to the accuracy and saves it after each epoch data we get
    seed=42,
    eval_strategy="epoch",
    save_strategy="epoch",
    # Set the best model with the best results and deletes the original model
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
)
# We call the data to train the model and then test it with other data
# after each epoch is done and we get the accuracy measure with compute_metrics
trainer_sst2 = Trainer(
    model=model_sst2,
    args=training_args_sst2,
    train_dataset=sst2_encoded["train"],
    eval_dataset=sst2_encoded["validation"],
    compute_metrics=compute_metrics_sst2,
)

trainer_sst2.train()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1087.31it/s]
BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint.

Epoch,Training Loss,Validation Loss


In [ ]:
# ── Part 5: Report final validation accuracy for SST-2 ───────────────────────
# We store this data here and we print the accuracies number.
sst2_results = trainer_sst2.evaluate()
print("=" * 50)
print("SST-2 Baseline — Final Validation Results")
print("=" * 50)
print(f"  Accuracy : {sst2_results['eval_accuracy']:.4f}")
print(f"  (Reference BERT paper: 93.5%)")

SST-2 Baseline — Final Validation Results
  Accuracy : 0.9278
  (Reference BERT paper: 93.5%)


---
## Fine-tune BERT on MRPC
We fine-tuned BERT on MRPC using a fresh copy of the model and the same baseline hyperparameters.




In [ ]:
# ── Part 6: Fine-tune on MRPC ────────────────────────────────────────────────

model_mrpc = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
)
# We created and stored a folder where we will have certain data that we get back
# prior to when we finish training the epochs
training_args_mrpc = TrainingArguments(
    output_dir="./results_mrpc",
    num_train_epochs=3,
    # We use 2e-5 to make it a certain size after calculating the weights
    # We set batch size in the sentences so that it knows how much it reads
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    # We set seed to 42 to make sure we get consistent results
    # We see to the accuracy and saves it after each epoch data we get
    seed=42,
    eval_strategy="epoch",
    save_strategy="epoch",
    # Set the best model with the best results and deletes the original model
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)
# We call the data to train the model and then test it with other data
# after each epoch is done and we get the accuracy measure with compute_metrics
trainer_mrpc = Trainer(
    model=model_mrpc,
    args=training_args_mrpc,
    train_dataset=mrpc_encoded["train"],
    eval_dataset=mrpc_encoded["validation"],
    compute_metrics=compute_metrics_mrpc,
)

trainer_mrpc.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.488683,0.794118,0.862295
2,No log,0.394037,0.821078,0.875214
3,No log,0.403209,0.828431,0.880137


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=345, training_loss=0.4179863086645154, metrics={'train_runtime': 60.8697, 'train_samples_per_second': 180.78, 'train_steps_per_second': 5.668, 'total_flos': 723818513295360.0, 'train_loss': 0.4179863086645154, 'epoch': 3.0})

In [ ]:
# ── Part 6: Report final validation accuracy and F1 for MRPC ─────────────────
# We store this data here and we print the accuracies number.
mrpc_results = trainer_mrpc.evaluate()
print("=" * 50)
print("MRPC Baseline — Final Validation Results")
print("=" * 50)
print(f"  Accuracy : {mrpc_results['eval_accuracy']:.4f}")
print(f"  F1       : {mrpc_results['eval_f1']:.4f}")
print(f"  (Reference BERT paper F1: 88.9%)")

MRPC Baseline — Final Validation Results
  Accuracy : 0.8284
  F1       : 0.8801
  (Reference BERT paper F1: 88.9%)


---
## Fine-tune BERT on SQuAD
We fine-tuned BERT on SQuAD using a fresh copy of the model and the same baseline hyperparameters.


In [ ]:
# ── Part 7: Fine-tune on SQuAD ────────────────────────────────────────────────

model_squad = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_CHECKPOINT
)
# We created and stored a folder where we will have certain data that we get back
# prior to when we finish training the epochs
args_squad = TrainingArguments(
    output_dir="./results_squad",
    num_train_epochs=3,
    # We use 2e-5 to make it a certain size after calculating the weights
    # We set batch size in the sentences so that it knows how much it reads
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    # We set seed to 42 to make sure we get consistent results
    # We see to the accuracy and saves it after each epoch data we get
    # We save the model at the end of every training session to epoch
    seed=42,
    save_strategy="epoch",
    report_to="none"
)
# We define the model and we implement
# the results of the Squad data to the model
trainer_squad = Trainer(
    model=model_squad,
    args=args_squad,
    train_dataset=squad_train_encoded,
)

# Start training sessions
print("Starting training on SQuAD...")
trainer_squad.train()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly init

Starting training on SQuAD...


Step,Training Loss
500,1.902907
1000,1.285296
1500,1.159848
2000,1.120964
2500,1.053553
3000,0.881751
3500,0.718790
4000,0.735639
4500,0.720673
5000,0.723969


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8301, training_loss=0.817459012697783, metrics={'train_runtime': 3669.663, 'train_samples_per_second': 72.37, 'train_steps_per_second': 2.262, 'total_flos': 5.204482670991974e+16, 'train_loss': 0.817459012697783, 'epoch': 3.0})

In [ ]:
# 1. Preprocess the validation set for evaluation

# We used the tokenizer and we use this to create questions and we put it all on a list
# We shorten and then create the pieces, then we use these pieces
def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    # We link these tokens back to its first char position.
    # We finish this by setting padding to max_length which
    # puts the empty tokens and have it fill it up so that we have
    # the set amount of tokens.
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
# We set an empty list of example_ids to store the numbers
# We also used .pop to cut the maps to their base paragraphs
    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []
# We used a for loop to loop through all sequences in using the index,
# We see the base number that coorelates to the same paragraph on the same
# sequence then we look up the string in the data and append it to the list
    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])
        # We inputs. and did what we did earlier by putting in a
        # bunch of zeroes and ones to get the current sequence by seeing where
        # the character maps are
        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        # We can extract these numbers from context with none for k,
        # this limits the chances of error by seeing the text, but rather if the
        # original token belongs to the prompt, then we add it back to the original data.
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]
    inputs["example_id"] = example_ids
    return inputs
# We preprocess this to the entire data by seeking validation of other data.
print("Processing SQuAD validation dataset...")
squad_val_encoded = squad["validation"].map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=squad["validation"].column_names,
)

# We finish generating the predicaments after finishing the other data through
# the model to get the chances of the starting strings and the ending strings.
print("Generating predictions...")
predictions, _, _ = trainer_squad.predict(squad_val_encoded)
start_logits, end_logits = predictions

# We use this to convert these numbers into strings
final_predictions = postprocess_qa_predictions(
    squad["validation"],
    squad_val_encoded,
    (start_logits, end_logits)
)

# We read the original dataset and have it compare to the other data.
references = [{"id": ex["id"], "answers": ex["answers"]} for ex in squad["validation"]]
# We use this to see the predicaments of the model and the actual manual answers
# Then we print both the exact match and f1 score.
print("=" * 50)
print("SQuAD Final Validation Results")
print("=" * 50)
results_squad = squad_metric.compute(predictions=final_predictions, references=references)
print(f"  Exact Match (EM) : {results_squad['exact_match']:.2f}%")
print(f"  F1 Score         : {results_squad['f1']:.2f}%")


Processing SQuAD validation dataset...


Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

Generating predictions...


Post-processing predictions:   0%|          | 0/10570 [00:00<?, ?it/s]

SQuAD Final Validation Results
  Exact Match (EM) : 80.77%
  F1 Score         : 88.41%


---
## SST-2 to MRPC
Fine-tune BERT on MRPC using the model already fine-tuned on SST-2 and the same baseline hyperparameters.

In [ ]:
# Transfer Learning (SST-2 -> MRPC) ─────────────────────────────────

# 1. Load the best SST-2 model checkpoint
# We use trainer_sst2.state.best_model_checkpoint to get the path to the saved SST-2 model
try:
    best_sst2_checkpoint = trainer_sst2.state.best_model_checkpoint
    print(f"Loading SST-2 fine-tuned model from: {best_sst2_checkpoint}")
except AttributeError:
    print("Warning: Could not find best SST-2 checkpoint in memory. Make sure you have run Part 5.")
    best_sst2_checkpoint = "./results_sst2"

model_transfer = AutoModelForSequenceClassification.from_pretrained(
    best_sst2_checkpoint,
    num_labels=2,
)

# 2. Training Arguments (Same baseline hyperparameters)
training_args_transfer = TrainingArguments(
    output_dir="./results_mrpc_transfer",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    seed=42,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)

# 3. Trainer for MRPC
trainer_transfer = Trainer(
    model=model_transfer,
    args=training_args_transfer,
    train_dataset=mrpc_encoded["train"],
    eval_dataset=mrpc_encoded["validation"],
    compute_metrics=compute_metrics_mrpc,
)

# 4. Train and Evaluate
print("\nStarting transfer learning on MRPC...")
trainer_transfer.train()

transfer_results = trainer_transfer.evaluate()
print("=" * 50)
print("MRPC Transfer (from SST-2) — Final Validation Results")
print("=" * 50)
print(f"  Accuracy : {transfer_results['eval_accuracy']:.4f}")
print(f"  F1       : {transfer_results['eval_f1']:.4f}")


Loading SST-2 fine-tuned model from: ./results_sst2/checkpoint-2105


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Starting transfer learning on MRPC...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.505837,0.781863,0.851419
2,No log,0.446591,0.825980,0.883415
3,No log,0.449653,0.821078,0.876481


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

MRPC Transfer (from SST-2) — Final Validation Results
  Accuracy : 0.8260
  F1       : 0.8834


---
## Visualization

### Interactive QA with Fine-tuned BERT

Instead of using the benchmark dataset, you can now provide any custom text context and ask your own questions to see how well the model extracts the correct answer from your text.

In [ ]:
from transformers import pipeline

# 1. Initialize the QA pipeline with our fine-tuned model and tokenizer
qa_tool = pipeline(
    "question-answering",
    model=model_squad,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1 # Use GPU if available
)

# 2. Define your OWN custom context and question
my_context = """
UGA is a school that is known for football.
"""

my_question = "What is UGA known for?"

print("========== CUSTOM CONTEXT ==========")
print(my_context.strip())
print("====================================\n")

print(f"[Question]: {my_question}")

# 3. Use the pipeline to get the answer
answer = qa_tool(question=my_question, context=my_context)

print(f"-> [Predicted Answer]: {answer['answer']} (Confidence: {answer['score']:.4f})")

# Feel free to change my_context and my_question to anything you want!

========== CUSTOM CONTEXT ==========
Google Colab is a hosted Jupyter notebook service that requires no setup to use, 
while providing free access to computing resources including GPUs. It is widely 
used for machine learning, data analysis, and education.

[Question]: What is Google Colab widely used for?
-> [Predicted Answer]: machine learning, data analysis, and education (Confidence: 0.8089)


### Interactive Inference for SST-2 and MRPC
Just like we did with SQuAD, we can wrap our SST-2 (Sentiment Analysis) and MRPC (Paraphrase Detection) fine-tuned models into interactive pipelines.

1. **SST-2 (Sentiment):** Update the input sentence in the code below to any English sentence you want to test (e.g., a movie or product review). The model will predict whether the overall sentiment is Positive or Negative.
2. **MRPC (Paraphrase):** Provide any two sentences of your choice in the appropriate strings below. The model will predict whether they share the exact same meaning (Equivalent) or not.


In [ ]:
from transformers import pipeline

# ==========================================
# 1. SST-2: Sentiment Classification
# ==========================================
print("\n" + "="*40)
print("SST-2: Sentiment Classification Tool")
print("="*40)
sst2_tool = pipeline(
    "text-classification",
    model=model_sst2,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# Define your own sentence for sentiment analysis
my_sentiment_sentence = "The sun is up so today is going to be the best day ever!"

print(f"[Input]: {my_sentiment_sentence}")
sst2_result = sst2_tool(my_sentiment_sentence)[0]

# Map LABEL_0 and LABEL_1 to readable text based on SST-2 definition
sst2_label_map = {"LABEL_0": "Negative", "LABEL_1": "Positive"}
pred_sentiment = sst2_label_map.get(sst2_result['label'], sst2_result['label'])
print(f"-> [Prediction]: {pred_sentiment} (Confidence: {sst2_result['score']:.4f})")


# ==========================================
# 2. MRPC: Paraphrase Detection
# ==========================================
print("\n" + "="*40)
print("MRPC: Paraphrase Detection Tool")
print("="*40)
# We'll use the model from transfer learning (model_transfer) for MRPC
mrpc_tool = pipeline(
    "text-classification",
    model=model_transfer,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# Define your own sentence pair to check if they mean the same thing
sentence_A = "I like to drink hot water when I wake up."
sentence_B = "I like to drink a cold lemonade when I go to sleep."

print(f"[Sentence 1]: {sentence_A}")
print(f"[Sentence 2]: {sentence_B}")

# Hugging Face pipeline accepts sentence pairs by passing text and text_pair
# Passing it inside a list guarantees a list of results is returned
mrpc_result = mrpc_tool([{"text": sentence_A, "text_pair": sentence_B}], truncation=True, max_length=128)[0]

# Map LABEL_0 and LABEL_1 to readable text based on MRPC definition
mrpc_label_map = {"LABEL_0": "Not Equivalent", "LABEL_1": "Equivalent"}
pred_paraphrase = mrpc_label_map.get(mrpc_result['label'], mrpc_result['label'])
print(f"-> [Prediction]: {pred_paraphrase} (Confidence: {mrpc_result['score']:.4f})")


SST-2: Sentiment Classification Tool
[Input]: I absolutely loved the new sci-fi movie, the visuals were stunning!
-> [Prediction]: Positive (Confidence: 0.9983)

MRPC: Paraphrase Detection Tool
[Sentence 1]: The quick brown fox jumps over the lazy dog.
[Sentence 2]: A fast brown fox leaps across a sleepy dog.
-> [Prediction]: Equivalent (Confidence: 0.6987)
